In [ ]:
# Import necessary libraries
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import pandas as pd
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
from collections import Counter
from tensorflow.keras.models import Model


In [ ]:
# Step 1: Define paths to folders (بدون نیاز به CSV)
train_image_path  = "../input/17flowerclasses/17flowerclasses/train/"   # فولدر آموزشی با زیرفولدرهای کلاس‌ها
test_image_path  = "../input/17flowerclasses/17flowerclasses/test/"   # فولدر تست با زیرفولدرهای کلاس‌ها


# Step 2: Data Augmentation and Preprocessing
IMG_SIZE = 224  # اندازه تصویر (برای MobileNetV2)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=35,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.3,
    zoom_range=0.4,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest',
    validation_split=0.2
)

# برای Test، بدون افزایش داده
test_datagen = ImageDataGenerator(rescale=1./255)

# Step 3: Create generators from directories
train_generator = train_datagen.flow_from_directory(
    train_image_path,  # مسیر فولدر train
    target_size=(IMG_SIZE, IMG_SIZE),  # تغییر اندازه تصاویر
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # برای طبقه‌بندی چندکلاسه (one-hot encoding خودکار)
    shuffle=True,  # مخلوط کردن داده‌ها
    subset='training'  # اگر validation_split استفاده کردید
)

val_generator = train_datagen.flow_from_directory(  # یا val_datagen اگر فولدر جدا دارید
    train_image_path ,  # اگر val_dir وجود ندارد، از train استفاده کنید
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,  # برای validation بهتر است shuffle نکنید
    subset='validation'
)
test_generator = test_datagen.flow_from_directory(
    test_image_path,  # مسیر فولدر train
    target_size=(IMG_SIZE, IMG_SIZE),  # تغییر اندازه تصاویر
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # برای طبقه‌بندی چندکلاسه (one-hot encoding خودکار)
    shuffle=False,  # مخلوط کردن داده‌ها
    )
# چک کردن کلاس‌ها (اختیاری)
print(" کلاس‌های شناسایی‌شده در آموزش:", train_generator.class_indices)  # خروجی: {'class0': 0, 'class1': 1, ...}
print(" کلاس‌های شناسایی‌شده در تست:", test_generator.class_indices)  # خروجی: {'class0': 0, 'class1': 1, ...}


In [ ]:
## ۲. نمایش چند عکس رندوم از هر ژنراتور

def plot_random_images(generator, num_images=6, cols=3):
    # یک بچ از داده‌ها رو بگیریم
    images, labels = next(generator)
    
    # انتخاب تصادفی چند تصویر
    indices = np.random.choice(len(images), size=min(num_images, len(images)), replace=False)
    
    rows = (num_images + cols - 1) // cols
    plt.figure(figsize=(15, 5 * rows))
    
    class_names = {v: k for k, v in generator.class_indices.items()}
    
    for i, idx in enumerate(indices):
        plt.subplot(rows, cols, i+1)
        plt.imshow(images[idx])
        label_idx = np.argmax(labels[idx])  # چون one-hot هست
        plt.title(f"{class_names[label_idx]}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# نمایش 6 تصویر رندوم از train
print("تصاویر رندوم از Train:")
plot_random_images(train_generator, num_images=6)

# از validation
print("تصاویر رندوم از Validation:")
plot_random_images(val_generator, num_images=6)

# از test
print("تصاویر رندوم از Test:")
plot_random_images(test_generator, num_images=6)


In [ ]:
# === بررسی توزیع کلاس‌ها ===
def get_class_distribution(generator):
    labels = generator.labels
    class_counts = Counter(labels)
    class_names = {v: k for k, v in generator.class_indices.items()}
    return {class_names[i]: class_counts[i] for i in class_counts}, generator.samples

train_dist, train_total = get_class_distribution(train_generator)
val_dist, val_total = get_class_distribution(val_generator)
test_dist, test_total = get_class_distribution(test_generator)

print(f"Train ({train_total}):", train_dist)
print(f"Val ({val_total}):", val_dist)
print(f"Test ({test_total}):", test_dist)


**ساخت مدل**

In [ ]:
# --- تنظیمات ---
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 17
EPOCHS = 20
EPOCHS_INITIAL = 50
EPOCHS_FINE = 40
# --- ساخت مدل ---

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)       
x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
predictions = Dense(17, activation='softmax')(x)
"""
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
# فریز کامل در ابتدا
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x) 
x = BatchNormalization()(x)
x = Dropout(0.5)(x)       
x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
predictions = Dense(17, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)
"""
# --- مرحله ۱: آموزش فقط لایه‌های بالا ---
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**آموزش مدل**

In [ ]:


# Callbacks ضد overfit
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=4, min_lr=1e-7, verbose=1)
]

print("آموزش لایه‌های بالا...")
history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_INITIAL,
    callbacks=callbacks,
    verbose=1
)
# --- آنفریز کردن فقط 28 لایه آخر ---
# 5. فاین‌تونینگ هوشمند (اینجاست که دقت می‌پره بالا!)
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 28  # فقط 28 لایه آخر
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=Adam(5e-6), loss='categorical_crossentropy', metrics=['accuracy'])  # lr خیلی کم!


print(f"فاین‌تونینگ از لایه {fine_tune_at} به بعد...")
history2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
    verbose=1
)


**ارزیابی روی تست**

In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f"\nدقت نهایی روی تست: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# --- تابع پیش‌بینی ---
def predict_flower(image_path, model, class_indices):
    # بارگذاری و پیش‌پردازش تصویر
    img = image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0

    # پیش‌بینی
    predictions = model.predict(img_array, verbose=0)
    predictions = predictions.flatten()  # (17,) → مطمئن بشیم یک‌بعدیه

    # معکوس کردن class_indices
    idx_to_class = {v: k for k, v in class_indices.items()}

    # پیش‌بینی نهایی
    predicted_idx = np.argmax(predictions)
    confidence = predictions[predicted_idx]
    predicted_class = idx_to_class[predicted_idx]

    # لیست مرتب‌شده احتمالات (۱۰ تای اول)
    sorted_indices = np.argsort(predictions)[::-1]
    
    print("\n" + "="*60)
    print(f"نتایج پیش‌بینی برای: {image_path.split('/')[-1]}")
    print("="*60)
    for rank, idx in enumerate(sorted_indices[:10], 1):
        name = idx_to_class[idx]
        prob = predictions[idx]
        star = "پیش‌بینی نهایی" if rank == 1 else ""
        print(f"{rank:2d}. {name:20} → {prob:.4f} ({prob:.2%}) {star}")
    print("="*60)

    return predicted_class, confidence, img
# --- استفاده ---
# class_indices از train_generator 
class_indices = train_generator.class_indices
for i in (1,2,3,4,5):
    # عکس خارجی
    external_image_path = "/kaggle/input/pictotest/image ("+str(i) +").jpg"  
    print(external_image_path)
    label, conf, img = predict_flower(external_image_path, model, class_indices)
    plt.imshow(img)
    plt.title(f"پیش‌بینی: {label}\nاعتماد: {conf:.2%}")
    plt.axis('off')
    plt.show()
    print(f"این گل احتمالاً: {label} است ({conf:.2%})")
